In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os
from pathlib import Path
import joblib
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import torch
from sklearn.metrics import accuracy_score,f1_score

In [2]:
put = Path('/content/drive/MyDrive/bert_data.parquet') # Написать путь где хранится bert_data
df = pd.read_parquet(put)

In [3]:
df_train,df_val = train_test_split(df,test_size=0.2,stratify=df['topic'],random_state= 42)

# Кодировка целевой переменной

In [4]:
enc = LabelEncoder() # кодирование категориальных данных
df_train['label'] = enc.fit_transform(df_train['topic'])
df_val['label'] = enc.transform(df_val['topic'])

# Модель

In [5]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(df_train['label'])
w = compute_class_weight('balanced', classes=classes, y=df_train['label'])
class_weights = torch.tensor(w, dtype=torch.float)

In [6]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [7]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=1)
  acc = accuracy_score(labels,predictions)
  f1_macro = f1_score(labels,predictions,average = 'macro')
  f1_weighted = f1_score(labels,predictions,average = 'weighted')
  return{'accuracy =':acc, 'f1 score(macro) =':f1_macro,'f1 score(weighted) =':f1_weighted
  }

In [8]:
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

## Параметры

In [9]:
training_args = TrainingArguments(output_dir='./bert_results',
    num_train_epochs=3,
    per_device_train_batch_size=48,
    per_device_eval_batch_size=64,
    learning_rate=1e-4,
    weight_decay =0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1 score(macro) =',
    fp16 = True,
    logging_steps=50,
    report_to='none'
)

## Токенизация

In [10]:
model_name = 'cointegrated/rubert-tiny2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_bert = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels = len(enc.classes_))
df_train = df_train.dropna(subset=['text'])

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  118MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
train_dataset = Dataset.from_pandas(df_train[['text', 'label']])
val_dataset = Dataset.from_pandas(df_val[['text', 'label']])
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [12]:
torch.cuda.is_available()

True

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_bert.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, 

## Обучение

In [14]:
trainer = WeightedTrainer(model = model_bert,args = training_args,train_dataset=train_dataset,eval_dataset=val_dataset,compute_metrics=compute_metrics)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy =,F1 score(macro) =,F1 score(weighted) =
1,0.671518,0.593225,0.782550,0.703371,0.789435
2,0.423159,0.599257,0.818850,0.752534,0.819800
3,0.239835,0.622727,0.820400,0.754069,0.822159


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5001, training_loss=0.5184529495344141, metrics={'train_runtime': 717.223, 'train_samples_per_second': 334.624, 'train_steps_per_second': 6.973, 'total_flos': 886520217600000.0, 'train_loss': 0.5184529495344141, 'epoch': 3.0})

## Результат лучшей эпохи

In [15]:
print('f1 score лучшей эпохи:',trainer.state.best_metric)

f1 score лучшей эпохи: 0.7540694490609683


# Сохранение модели и токенов

In [18]:
put = Path('/content/drive/MyDrive/') # написать свой путь для сохранения моделей
trainer.save_model(put/'bert_finetuned')
tokenizer.save_pretrained(put/'bert_token_finetuned')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/bert_token_finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/bert_token_finetuned/tokenizer.json')